# Zerlaut parameter exploration

## Initialization

**Environment**

In [ ]:
from dotenv import dotenv_values, find_dotenv
from pathlib import Path
import os, sys

# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))

**Modules**

In [ ]:
import json5 as json
from copy import deepcopy

import pandas as pd

from IPython.display import display
# from itables import show
# from ipydatagrid import DataGrid
from ipyaggrid import Grid


In [ ]:
from aux.environment import resolve_directory

**Console**

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

**Set main directory paths**

In [ ]:
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Load parameters and groups

In [ ]:
CONFIG_PATH = WORK_DIR / 'config' / 'zerlaut' / 'params_raw.json'

pprint(f'Parameter configuration: {CONFIG_PATH}')

In [ ]:
GROUPS = {
    'Membrane properties': ['g_L', 'E_L_e', 'E_L_i', 'C_m'],
    'Adaptation': ['b_e', 'a_e', 'b_i', 'a_i', 'tau_w_e', 'tau_w_i'],
    'Synaptic properties': ['E_e', 'E_i', 'Q_e', 'Q_i', 'tau_e', 'tau_i'],
    'Network properties': ['N_tot', 'p_connect_e', 'p_connect_i', 'g', 'K_ext_e', 'K_ext_i', 'T'],
    'Phenomenological threshold coefficients': ['P_e', 'P_i'],
    'External inputs': [
        'external_input_ex_ex', 'external_input_ex_in',
        'external_input_in_ex', 'external_input_in_in',
    ],
    'Noise': ['tau_OU', 'weight_noise'],
    'Inhibitory coupling scaling': ['S_i'],
}

In [ ]:
with CONFIG_PATH.open(encoding='utf-8') as config_file:
    PARAMS = json.load(config_file)

pprint(f'Loaded {len(PARAMS)} parameter names in {len(GROUPS)} groups.')

## Show paramters

In [ ]:
def make_parameter_table(params, groups):
    group_lookup = {
        name: group
        for group, names in groups.items()
        for name in names
    }
    return pd.DataFrame([
        {'Parameter': name, **data, 'Group': group_lookup.get(name)}
        for name, data in params.items()
    ])

def show_table(df):
   display(Grid(
        grid_data=df,
        theme="ag-theme-balham-dark",
        grid_options={
            "defaultColDef": {"filter": True, "resizable": True, 'editable': False,},
            #'columnDefs': [],
            'pagination': True,
            'paginationPageSize': 50,
        },
        quick_filter=True,
        columns_fit="auto"
    ))


In [ ]:
parameter_table = make_parameter_table(PARAMS, GROUPS)
print('Zerlaut parameters and their default values:')
#show(parameter_table)
# display(parameter_table)
# show(parameter_table, columnControl=["order", "searchDropdown"], autoWidth=False)
# display(DataGrid(parameter_table))
show_table(parameter_table)


**Parameter counts**

In [ ]:
parameter_counts = parameter_table['default'].map(
    lambda value: len(value) if isinstance(value, list) else 1
)
counts_by_group = parameter_counts.groupby(parameter_table['Group'], sort=False).sum()

pprint('Parameters per group:', counts_by_group.to_dict())
pprint(f'Total parameters: {parameter_counts.sum()}')
pprint(f"Excluding external inputs: {parameter_counts[parameter_table['Group'] != 'External inputs'].sum()}")
pprint(f"Excluding external inputs and noise: {parameter_counts[~parameter_table['Group'].isin(['External inputs', 'Noise'])].sum()}")

## Functions to adjust parameter ranges

In [ ]:
def bounds_from_default(params, names, margin=0.10):
    ranges = deepcopy(params)
    for name in names:
        default = params[name]['default']
        
        for key, sign in [('MIN', -1), ('MAX', 1)]:
            def adjust(v): return v + sign * abs(v) * margin
            
            ranges[name][key] = (
                [adjust(v) for v in default]
                if isinstance(default, list) else adjust(default)
            )
    return ranges

In [ ]:
def expand_bounds (params, names, margin=0.10):
    if margin < 0:
        raise ValueError('Variation must be non-negative.')

    ranges = deepcopy(params)
    for name in names:
        for key, sign in [('MIN', -1), ('MAX', 1)]:
            def adjust(v): return v + sign * abs(v) * margin

            value = params[name][key]
            ranges[name][key] = (
                [adjust(v) for v in value]
                if isinstance(value, list) else adjust(value)
            )
    return ranges

